In [1]:
import numpy as np
import pandas as pd
import pathlib
from tqdm.auto import tqdm

import hydra
from omegaconf import DictConfig, OmegaConf

import torch
import ray

In [2]:
node = !hostname
if "sc" in node[0]:
    base_path = "/sc-projects/sc-proj-ukb-cvd"
else: 
    base_path = "/data/analysis/ag-reils/ag-reils-shared/cardioRS"
print(base_path)

project_label = "22_medical_records"
project_path = f"{base_path}/results/projects/{project_label}"
figure_path = f"{project_path}/figures"
output_path = f"{project_path}/data"

pathlib.Path(figure_path).mkdir(parents=True, exist_ok=True)
pathlib.Path(output_path).mkdir(parents=True, exist_ok=True)

experiment = 230425
experiment_path = f"{output_path}/{experiment}"
pathlib.Path(experiment_path).mkdir(parents=True, exist_ok=True)

/sc-projects/sc-proj-ukb-cvd


In [ ]:
ray.shutdown()

In [3]:
ray.init(num_cpus=24, include_dashboard=False)

2023-05-10 11:11:38,038	INFO worker.py:1529 -- Started a local Ray instance. View the dashboard at 127.0.0.1:8267 


Python version:,3.10.9
Ray version:,2.2.0
Dashboard:,http://127.0.0.1:8267


In [7]:
import wandb
api = wandb.Api(timeout=120)
entity, project = "cardiors", "recordgraphs"  # set to your entity and project 
tag = "230428_covid_inference"
runs = api.runs(entity + "/" + project, filters={"tags": {"$in": [tag]}}) 

In [11]:
len(runs)

22

In [9]:
run_list = []
for run in tqdm(runs):
    run_list.append(
        {
            "id": run.path[-1], 
            "name": run.name,
            "tags": run.tags,
            "config": {k: v for k,v in run.config.items() if not k.startswith('_')},
            "summary": run.summary._json_dict,
            "path": None if "best_checkpoint" not in run.config.keys() else str(pathlib.Path(run.config["best_checkpoint"]).parent.parent)
        }
    )

Exception ignored in: <function tqdm.__del__ at 0x7fac51536680>
Traceback (most recent call last):
  File "/home/steinfej/miniconda3/envs/udm/lib/python3.10/site-packages/tqdm/std.py", line 1162, in __del__
    self.close()
  File "/home/steinfej/miniconda3/envs/udm/lib/python3.10/site-packages/tqdm/notebook.py", line 282, in close
    if self.disable:
AttributeError: 'tqdm' object has no attribute 'disable'


  0%|          | 0/22 [00:00<?, ?it/s]

In [ ]:
model_path = "/sc-projects/sc-proj-ukb-cvd/results/models/RecordGraphs/"

In [12]:
pd.DataFrame(run_list)

,id,name,tags,config,summary,path
0,133al0d6,autumn-darkness-6444,"[230428_covid_inference, full_data, identity]","{'feature_metadata': {'records': {'n': 15596, ...",{'_wandb': {'runtime': 769}},None
1,19ntedgh,scarlet-donkey-6443,"[230428_covid_inference, full_data, identity]","{'feature_metadata': {'records': {'n': 15596, ...",{'_wandb': {'runtime': 878}},None
2,1u8pz9b9,winter-firefly-6442,"[230428_covid_inference, full_data, identity]","{'feature_metadata': {'records': {'n': 15596, ...",{'_wandb': {'runtime': 857}},None
3,1cblrjri,swift-armadillo-6441,"[230428_covid_inference, full_data, identity]","{'feature_metadata': {'records': {'n': 15596, ...",{'_wandb': {'runtime': 771}},None
4,3sjfxviz,polar-night-6440,"[230428_covid_inference, full_data, identity]","{'feature_metadata': {'records': {'n': 15596, ...",{'_wandb': {'runtime': 628}},None
5,1s8uwf1p,vivid-dragon-6439,"[230428_covid_inference, full_data, identity]","{'feature_metadata': {'records': {'n': 15596, ...",{'_wandb': {'runtime': 614}},None
6,13x5yibt,gentle-deluge-6438,"[230428_covid_inference, full_data, identity]","{'feature_metadata': {'records': {'n': 15596, ...",{'_wandb': {'runtime': 625}},None
7,z7q7jhj4,drawn-plasma-6437,"[230428_covid_inference, full_data, identity]","{'feature_metadata': {'records': {'n': 15596, ...",{'_wandb': {'runtime': 776}},None
8,2udx8zye,vibrant-music-6436,"[230428_covid_inference, full_data, identity]","{'feature_metadata': {'records': {'n': 15596, ...",{'_wandb': {'runtime': 773}},None
9,2j1w29pr,easy-firefly-6435,"[230428_covid_inference, full_data, identity]","{'feature_metadata': {'records': {'n': 15596, ...",{'_wandb': {'runtime': 852}},None


In [22]:
runs_df = pd.DataFrame(run_list)#query("path==path")
runs_df["path"] = "/sc-projects/sc-proj-ukb-cvd/results/models/RecordGraphs/" + runs_df["id"]
print(runs_df.shape)
runs_df.sort_values("name")

(22, 6)


,id,name,tags,config,summary,path
0,133al0d6,autumn-darkness-6444,"[230428_covid_inference, full_data, identity]","{'feature_metadata': {'records': {'n': 15596, ...",{'_wandb': {'runtime': 769}},/sc-projects/sc-proj-ukb-cvd/results/models/Re...
16,1nyo51rr,blooming-grass-6428,"[230428_covid_inference, full_data, identity]","{'feature_metadata': {'records': {'n': 15596, ...",{'_wandb': {'runtime': 760}},/sc-projects/sc-proj-ukb-cvd/results/models/Re...
19,2tt3rkyw,cool-deluge-6425,"[230428_covid_inference, full_data, identity]","{'feature_metadata': {'records': {'n': 15596, ...",{'_wandb': {'runtime': 624}},/sc-projects/sc-proj-ukb-cvd/results/models/Re...
14,2tpi4pvi,dauntless-armadillo-6430,"[230428_covid_inference, full_data, identity]","{'feature_metadata': {'records': {'n': 15596, ...",{'_wandb': {'runtime': 758}},/sc-projects/sc-proj-ukb-cvd/results/models/Re...
7,z7q7jhj4,drawn-plasma-6437,"[230428_covid_inference, full_data, identity]","{'feature_metadata': {'records': {'n': 15596, ...",{'_wandb': {'runtime': 776}},/sc-projects/sc-proj-ukb-cvd/results/models/Re...
9,2j1w29pr,easy-firefly-6435,"[230428_covid_inference, full_data, identity]","{'feature_metadata': {'records': {'n': 15596, ...",{'_wandb': {'runtime': 852}},/sc-projects/sc-proj-ukb-cvd/results/models/Re...
11,389twhxs,expert-lion-6433,"[230428_covid_inference, full_data, identity]","{'feature_metadata': {'records': {'n': 15596, ...",{'_wandb': {'runtime': 631}},/sc-projects/sc-proj-ukb-cvd/results/models/Re...
15,28yxnxnk,fresh-field-6429,"[230428_covid_inference, full_data, identity]","{'feature_metadata': {'records': {'n': 15596, ...",{'_wandb': {'runtime': 800}},/sc-projects/sc-proj-ukb-cvd/results/models/Re...
6,13x5yibt,gentle-deluge-6438,"[230428_covid_inference, full_data, identity]","{'feature_metadata': {'records': {'n': 15596, ...",{'_wandb': {'runtime': 625}},/sc-projects/sc-proj-ukb-cvd/results/models/Re...
17,20hlbiwn,lilac-monkey-6427,"[230428_covid_inference, full_data, identity]","{'feature_metadata': {'records': {'n': 15596, ...",{'_wandb': {'runtime': 859}},/sc-projects/sc-proj-ukb-cvd/results/models/Re...


In [21]:
runs_df["path"].iloc[0]

'/sc-projects/sc-proj-ukb-cvd/results/models/RecordGraphs/133al0d6/predictions/predictions.feather'

In [12]:
runs_df.iloc[0].path

'/sc-projects/sc-proj-ukb-cvd/results/models/RecordGraphs/w0vw9p4h'

In [14]:
temp = pd.read_feather("/sc-projects/sc-proj-ukb-cvd/results/models/RecordGraphs/133al0d6/predictions/predictions.feather")

In [15]:
temp

,index,OMOP_4306655,phecode_001,phecode_002,phecode_002-1,phecode_003,phecode_004,phecode_004-1,phecode_004-2,phecode_004-3,...,phecode_979,phecode_979-2,phecode_979-7,partition,module,encoder,head,covariate_cols,record_cols,split
0,4196644,-3.023430,0.392832,-1.079083,-0.989298,-0.013465,-0.815178,-0.938865,-1.420820,-1.378703,...,-1.391467,-0.650496,-2.569006,6,RecordsIdentityTraining,None,MLPHead,[],"['OMOP_1000560', 'OMOP_1000632', 'OMOP_1000995...",train
1,3678640,-1.085831,-0.748256,-1.151758,-1.208433,-0.503294,-1.148521,-1.725332,-0.890989,-0.978545,...,-1.137585,-0.151188,-2.448870,6,RecordsIdentityTraining,None,MLPHead,[],"['OMOP_1000560', 'OMOP_1000632', 'OMOP_1000995...",train
2,3542454,-1.573979,-0.579084,-1.005102,-0.980639,-0.861540,-1.141987,-1.134000,-0.956642,-1.513900,...,-0.707294,0.477993,-2.035967,6,RecordsIdentityTraining,None,MLPHead,[],"['OMOP_1000560', 'OMOP_1000632', 'OMOP_1000995...",train
3,5230713,-0.571794,-0.806532,-1.498115,-1.509850,-0.692266,-0.936145,-1.373790,-1.102478,-1.737673,...,-1.303061,-1.371436,-0.755044,6,RecordsIdentityTraining,None,MLPHead,[],"['OMOP_1000560', 'OMOP_1000632', 'OMOP_1000995...",train
4,1980056,-4.097597,0.367956,-1.418018,-1.334728,-0.822585,-1.098619,-1.467062,-1.040672,-1.879112,...,-2.286474,-1.505977,-2.592870,6,RecordsIdentityTraining,None,MLPHead,[],"['OMOP_1000560', 'OMOP_1000632', 'OMOP_1000995...",train
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
478098,6024953,-3.969854,0.328685,-1.604424,-1.620170,-1.325243,-0.874249,-1.048770,-0.839142,-1.610433,...,-1.802008,-1.827339,-0.854436,6,RecordsIdentityTraining,None,MLPHead,[],"['OMOP_1000560', 'OMOP_1000632', 'OMOP_1000995...",test
478099,6024971,-2.252863,0.224587,-1.239970,-1.234634,-0.507474,-0.669180,-0.276952,-1.346914,-1.891803,...,-1.200621,-1.811466,-0.652684,6,RecordsIdentityTraining,None,MLPHead,[],"['OMOP_1000560', 'OMOP_1000632', 'OMOP_1000995...",test
478100,6025001,-2.186860,0.526358,0.043661,0.151743,-0.404995,-0.224563,-0.154687,-0.408305,-0.448138,...,0.161320,0.352546,-1.614432,6,RecordsIdentityTraining,None,MLPHead,[],"['OMOP_1000560', 'OMOP_1000632', 'OMOP_1000995...",test
478101,6025034,-2.526110,0.810040,-0.319046,-0.171159,-0.586102,-0.395552,-0.290282,-0.984672,-0.901149,...,-1.167964,-1.304386,-1.515227,6,RecordsIdentityTraining,None,MLPHead,[],"['OMOP_1000560', 'OMOP_1000632', 'OMOP_1000995...",test


## Process Predictions

In [16]:
name_dict = {
    "CovariatesOnlyTraining_['age_at_recruitment_f21022_0_0', 'sex_f31_0_0']_None_MLPHead": "Identity(AgeSex)+MLP",
    "RecordsIdentityTraining_[]_None_MLPHead": "Identity(Records)+MLP",
     "RecordsIdentityTraining_[]_None_LinearHead": "Identity(Records)+Linear",
    "RecordsGraphTraining_[]_HeteroGNN_MLPHead": "GNN(Records)+MLP",
    "RecordsIdentityTraining_['age_at_recruitment_f21022_0_0', 'sex_f31_0_0']_None_MLPHead": "Identity(AgeSex+Records)+MLP",
    "RecordsGraphTraining_['age_at_recruitment_f21022_0_0', 'sex_f31_0_0']_HeteroGNN_MLPHead": "GNN(AgeSex+Records)+MLP"
}

In [17]:
id_vars = ["eid", "model", "partition", "split"]

In [18]:
out_path = f"{experiment_path}/loghs_covid"
pathlib.Path(out_path).mkdir(parents=True, exist_ok=True)

In [23]:
@ray.remote
def prepare_predictions(in_path, out_path):
    
    temp = pd.read_feather(in_path).rename(columns={"index": "eid"}).reset_index(drop=True)
    temp["model"] = (temp.module.astype(str) + "_" + temp.covariate_cols.astype(str) + "_" + temp.encoder.astype(str) + "_" + temp["head"].astype(str)).astype("category")
    temp = temp.replace({"model":name_dict}).drop(columns=["module", "encoder", "head", "covariate_cols", "record_cols"])
    for c in id_vars: temp[c] = temp[c].astype("category")
    
    model = temp.model.unique()[0]
    partition = temp.partition.unique()[0]
    for split in ["train", "valid", "test"]:
        fp_out = f"{out_path}/{model}/{partition}"
        pathlib.Path(fp_out).mkdir(parents=True, exist_ok=True)
        temp.query("split==@split").reset_index(drop=True).to_feather(f"{fp_out}/{split}.feather")

In [24]:
for p in tqdm(runs_df.path): 
    prepare_predictions.remote(f"{p}/predictions/predictions.feather", out_path)

  0%|          | 0/22 [00:00<?, ?it/s]

In [ ]:
1+1